# Medicare Enrollment Cleaning

This script takes in the most recent Medicare Enrollment data download from CMS. In this case, it is June 2025. This has data 
from 2021 to 2023.


Data Cleaning Steps:
- Change all data to columns to snake case
- Use the accompanying data dictionary to change the data types of specified columns
- filter dataset only at the county and year level. 
- keep only totals per county. 
- Remove unknown counties.
- Impute the beneficiary enrollment data. Returns the median if there are at values for that county in at least 2 years. If not, return a static number of 5. Entries are hidden under 10 so imputing static number of 5. 


Move the final dataset to /dsa/groups/casestudycf25/team02/medicare_enrollment_clean.csv

In [1]:
# Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Change all headers to snake case
import re

def to_snake_case(name: str) -> str:
    # Add underscore between lower-to-upper transitions
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Replace non-alphanumeric with underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)

    # Remove leading/trailing underscores and lowercase
    return name.strip("_").lower()

In [3]:
# Read in CSV from file path
df = pd.read_csv("/dsa/groups/casestudycf25/team02/bronze/Medicare_Monthly_Enrollment_Jun_2025.csv")

df = df.rename(columns={col: to_snake_case(col) for col in df.columns})

df.head(5)

,year,month,bene_geo_lvl,bene_state_abrvtn,bene_state_desc,bene_county_desc,bene_fips_cd,tot_benes,orgnl_mdcr_benes,ma_and_oth_benes,...,b_tot_benes,b_orgnl_mdcr_benes,b_ma_and_oth_benes,prscrptn_drug_tot_benes,prscrptn_drug_pdp_benes,prscrptn_drug_mapd_benes,prscrptn_drug_deemed_eligible_full_lis_benes,prscrptn_drug_full_lis_benes,prscrptn_drug_partial_lis_benes,prscrptn_drug_no_lis_benes
0,2021,Year,National,US,National,Total,,"63,892,626","36,356,380","27,536,246",...,"58,376,622","30,842,701","27,533,921","48,818,849","24,164,736","24,654,113","11,711,263","1,108,918","323,250","35,675,418"
1,2021,Year,State,AL,Alabama,Total,01,"1,070,474","528,983","541,491",...,"997,828","456,357","541,470","806,300","290,817","515,483","218,199","32,582","8,713","546,806"
2,2021,Year,County,AL,Alabama,Autauga County,01001,"11,396","5,338","6,059",...,"10,608","4,549","6,059","7,838","1,931","5,907","2,072",371,93,"5,302"
3,2021,Year,County,AL,Alabama,Baldwin County,01003,"57,509","28,252","29,257",...,"53,916","24,660","29,256","43,320","15,447","27,873","6,350","1,047",338,"35,586"
4,2021,Year,County,AL,Alabama,Barbour County,01005,"6,200","2,658","3,542",...,"5,915","2,373","3,542","4,947","1,543","3,404","1,884",208,65,"2,791"


In [4]:
print(f"Original Dataset Shape: {df.shape}")

Original Dataset Shape: (130130, 54)


In [5]:
###################################
# filter dataset only at the county and year level. And keep only totals per county. Remove unknown counties.
###################################

# Keep County/Year level data
df_filtered = df[df["bene_geo_lvl"]  == "County"].copy()

# Only keep the Year totals by County
df_filtered = df_filtered[df_filtered["month"]  == "Year"].copy()

# Remove where there are unknown counties
df_filtered = df_filtered[df_filtered["bene_county_desc"]  != "Unknown"].copy()

# Remove counties where the entire county beneficiaries are unknown
df_filtered = df_filtered[df_filtered["tot_benes"]  != "*"].copy() 

In [6]:
print(f"New Dataset Shape After Filtering: {df_filtered.shape}")

New Dataset Shape After Filtering: (9671, 54)


In [7]:
# Drop columns
df_filtered.drop(columns = ['bene_geo_lvl', 'month'], inplace = True) # drop columns that have one value

# drop unnecessary
df_filtered.drop(columns = ['dsbld_esrd_and_esrd_only_benes', 'dsbld_esrd_and_esrd_only_benes',
                           'full_dual_tot_benes', 'part_dual_tot_benes', 'nodual_tot_benes'], inplace = True) 

df_filtered = df_filtered.drop([c for c in df.columns if c.startswith("a_")], axis=1) # drop Medicare Part A columns

df_filtered = df_filtered.drop([c for c in df.columns if c.startswith("prscrptn")], axis=1) # drop prescription columns

In [8]:
df_filtered.dtypes

year                    int64
bene_state_abrvtn      object
bene_state_desc        object
bene_county_desc       object
bene_fips_cd           object
tot_benes              object
orgnl_mdcr_benes       object
ma_and_oth_benes       object
aged_tot_benes         object
aged_esrd_benes        object
aged_no_esrd_benes     object
dsbld_tot_benes        object
dsbld_no_esrd_benes    object
male_tot_benes         object
female_tot_benes       object
white_tot_benes        object
black_tot_benes        object
api_tot_benes          object
hspnc_tot_benes        object
natind_tot_benes       object
othr_tot_benes         object
age_lt_25_benes        object
age_25_to_44_benes     object
age_45_to_64_benes     object
age_65_to_69_benes     object
age_70_to_74_benes     object
age_75_to_79_benes     object
age_80_to_84_benes     object
age_85_to_89_benes     object
age_90_to_94_benes     object
age_gt_94_benes        object
dual_tot_benes         object
b_tot_benes            object
b_orgnl_md

In [9]:
# Change column types
cols_to_string = [
    "bene_state_abrvtn",
    "bene_state_desc",
    "bene_county_desc",
    "bene_fips_cd"
]

df_filtered[cols_to_string] = df_filtered[cols_to_string].astype("str")

In [10]:
df_filtered = df_filtered.replace("*", np.nan)   # replace hidden entries with null

In [11]:
# Select columns by index: columns 5 through the end
cols = df_filtered.columns[5:]
cols

Index(['tot_benes', 'orgnl_mdcr_benes', 'ma_and_oth_benes', 'aged_tot_benes',
       'aged_esrd_benes', 'aged_no_esrd_benes', 'dsbld_tot_benes',
       'dsbld_no_esrd_benes', 'male_tot_benes', 'female_tot_benes',
       'white_tot_benes', 'black_tot_benes', 'api_tot_benes',
       'hspnc_tot_benes', 'natind_tot_benes', 'othr_tot_benes',
       'age_lt_25_benes', 'age_25_to_44_benes', 'age_45_to_64_benes',
       'age_65_to_69_benes', 'age_70_to_74_benes', 'age_75_to_79_benes',
       'age_80_to_84_benes', 'age_85_to_89_benes', 'age_90_to_94_benes',
       'age_gt_94_benes', 'dual_tot_benes', 'b_tot_benes',
       'b_orgnl_mdcr_benes', 'b_ma_and_oth_benes'],
      dtype='object')

In [12]:
# Update columns 5 through the end to be float type
df_filtered[cols] = (df_filtered[cols]
                            .replace({",": ""}, regex=True)   # remove commas
                            .astype(float)                    # convert to float
                    )

In [13]:
# Null values percent
nan_pct = df_filtered[cols].isna().mean() * 100

# Average of non-null values
non_null_mean = df_filtered[cols].mean(skipna=True)

# Combine
pre_impute= pd.DataFrame({
    "nan_pct": round(nan_pct,1),
    "mean_non_null": round(non_null_mean,0)
})

In [14]:
# drop highly suppressed cols
df_filtered.drop(columns = ['aged_esrd_benes', 'aged_no_esrd_benes', 'dsbld_no_esrd_benes', 
                            'white_tot_benes', 'black_tot_benes', 'api_tot_benes', 'hspnc_tot_benes', 'natind_tot_benes', 'othr_tot_benes']
                 , inplace = True) 


df_before_imputation = df_filtered.copy()

######################
## Impute the data
#####################

Impute the beneficiary enrollment data. Returns the median if there are at values for that county in at least 2 years. If not, return a static number of 5. Entries are hidden under 10 so imputing static number of 5. 

In [15]:
def impute_group(series):
    valid_count = series.notna().sum()

    if valid_count >= 2:
        return series.fillna(series.median()) # Return median if there are 2 or more datapoints for the county fips code
    else:
        return series.fillna(5) # Else return static number 5 (Entires are hidden if under 10 or countersuppressed)

In [16]:
# Select columns by index: columns 5 through the end
cols = df_filtered.columns[5:]
cols

# Apply imputation to all selected columns, grouped by fips
df_filtered[cols] = df_filtered.groupby("bene_fips_cd")[cols].transform(impute_group)

In [17]:
df_filtered[df_filtered["bene_county_desc"] == "Kenedy County"]

,year,bene_state_abrvtn,bene_state_desc,bene_county_desc,bene_fips_cd,tot_benes,orgnl_mdcr_benes,ma_and_oth_benes,aged_tot_benes,dsbld_tot_benes,...,age_70_to_74_benes,age_75_to_79_benes,age_80_to_84_benes,age_85_to_89_benes,age_90_to_94_benes,age_gt_94_benes,dual_tot_benes,b_tot_benes,b_orgnl_mdcr_benes,b_ma_and_oth_benes
2742,2021,TX,Texas,Kenedy County,48261,54.0,31.0,23.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,14.0,50.0,27.0,23.0
46111,2022,TX,Texas,Kenedy County,48261,52.0,27.0,24.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,14.0,49.0,25.0,24.0
89492,2023,TX,Texas,Kenedy County,48261,52.0,24.0,28.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,14.0,50.0,22.0,28.0


In [18]:
df_after_imputation = df_filtered.copy()

In [19]:
print(f"New Dataset Shape After Dropping Columns: {df_filtered.shape}")

New Dataset Shape After Dropping Columns: (9671, 26)


In [20]:
df_filtered.dtypes

year                    int64
bene_state_abrvtn      object
bene_state_desc        object
bene_county_desc       object
bene_fips_cd           object
tot_benes             float64
orgnl_mdcr_benes      float64
ma_and_oth_benes      float64
aged_tot_benes        float64
dsbld_tot_benes       float64
male_tot_benes        float64
female_tot_benes      float64
age_lt_25_benes       float64
age_25_to_44_benes    float64
age_45_to_64_benes    float64
age_65_to_69_benes    float64
age_70_to_74_benes    float64
age_75_to_79_benes    float64
age_80_to_84_benes    float64
age_85_to_89_benes    float64
age_90_to_94_benes    float64
age_gt_94_benes       float64
dual_tot_benes        float64
b_tot_benes           float64
b_orgnl_mdcr_benes    float64
b_ma_and_oth_benes    float64
dtype: object

In [21]:
df_filtered.tail(5)

,year,bene_state_abrvtn,bene_state_desc,bene_county_desc,bene_fips_cd,tot_benes,orgnl_mdcr_benes,ma_and_oth_benes,aged_tot_benes,dsbld_tot_benes,...,age_70_to_74_benes,age_75_to_79_benes,age_80_to_84_benes,age_85_to_89_benes,age_90_to_94_benes,age_gt_94_benes,dual_tot_benes,b_tot_benes,b_orgnl_mdcr_benes,b_ma_and_oth_benes
90076,2023,PR,Puerto Rico,Yabucoa Municipio,72151,9079.0,1131.0,7948.0,6385.0,2695.0,...,1714.0,1404.0,848.0,401.0,186.0,89.0,224.0,8077.0,131.0,7945.0
90077,2023,PR,Puerto Rico,Yauco Municipio,72153,9015.0,1143.0,7873.0,7694.0,1321.0,...,1949.0,1659.0,1079.0,623.0,223.0,89.0,192.0,8093.0,229.0,7864.0
90080,2023,VI,Virgin Islands,St. Croix Island,78010,9986.0,7144.0,2842.0,9307.0,679.0,...,2395.0,2257.0,1420.0,683.0,223.0,82.0,278.0,9190.0,6348.0,2842.0
90081,2023,VI,Virgin Islands,St. John Island,78020,935.0,740.0,195.0,892.0,44.0,...,248.0,212.0,113.0,50.0,5.0,5.0,20.0,862.0,667.0,195.0
90082,2023,VI,Virgin Islands,St. Thomas Island,78030,9754.0,6625.0,3129.0,9149.0,604.0,...,2356.0,2152.0,1328.0,629.0,274.0,92.0,237.0,8814.0,5686.0,3129.0


### Check the Null Values after Imputation

In [22]:
# Null values percent
nan_pct = df_filtered[cols].isna().mean() * 100

# Average of non-null values
non_null_mean = df_filtered[cols].mean(skipna=True)

# Combine
after_impute= pd.DataFrame({
    "nan_pct": round(nan_pct, 1),
    "mean_non_null": round(non_null_mean,0)
})


summary = pre_impute.merge(after_impute, 
                           left_index=True, 
                           right_index=True, 
                           how="inner", 
                           suffixes=("_before", "_after"))
summary["mean_diff"] = summary["mean_non_null_before"] - summary["mean_non_null_after"] 
display(summary)

,nan_pct_before,mean_non_null_before,nan_pct_after,mean_non_null_after,mean_diff
tot_benes,0.0,20046.0,0.0,20046.0,0.0
orgnl_mdcr_benes,0.9,10883.0,0.0,10790.0,93.0
ma_and_oth_benes,0.9,9332.0,0.0,9252.0,80.0
aged_tot_benes,0.9,17799.0,0.0,17645.0,154.0
dsbld_tot_benes,0.9,2421.0,0.0,2400.0,21.0
male_tot_benes,0.0,9146.0,0.0,9143.0,3.0
female_tot_benes,0.0,10907.0,0.0,10903.0,4.0
age_lt_25_benes,48.7,49.0,0.0,27.0,22.0
age_25_to_44_benes,51.6,898.0,0.0,441.0,457.0
age_45_to_64_benes,3.8,1968.0,0.0,1893.0,75.0


In [23]:
##############################################################
# Return a dataset that is clean for modeling use
##############################################################

df_filtered.to_csv("/dsa/groups/casestudycf25/team02/silver/medicare_enrollment_clean.csv", index=False)
df_filtered.shape

(9671, 26)

In [24]:
# Check the newly created CSV
temp = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/medicare_enrollment_clean.csv")

temp.shape

(9671, 26)